# 02 Predictive Track

Track A treats SparseTap as next-bit prediction. We run statistical lag scans, Logistic Regression with L1, XGBoost or fallback boosting, and a lightweight MLP. Every method saves artifacts and candidate rule hints.

In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

SEARCH_ROOTS = [Path.cwd().resolve(), Path.cwd().resolve().parent]
PROJECT_ROOT = next(path for path in SEARCH_ROOTS if (path / "DAY2_data.txt").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from sparsetap_config import DEFAULT_CONFIG
from sparsetap_utils import (
    build_supervised_dataset,
    candidate_table,
    prepare_environment,
    run_logistic_l1,
    run_mlp,
    run_pair_scan,
    run_single_lag_scan,
    run_xgboost,
    save_candidates,
    save_table,
)

config = DEFAULT_CONFIG
data, prefix = prepare_environment(config)
dataset = build_supervised_dataset(data, max_lag=config.max_lag)

single = run_single_lag_scan(data, prefix, config)
pair = run_pair_scan(data, prefix, [cand["taps"][0] for cand in single["candidates"][:12]], config)
logistic = run_logistic_l1(dataset, data, prefix, config)
xgb = run_xgboost(dataset, data, prefix, config)
mlp = run_mlp(dataset, data, prefix, config)

predictive_candidates = single["candidates"] + pair["candidates"] + logistic + xgb + mlp
save_candidates(predictive_candidates, config.candidate_dir / "predictive_candidates.json")
predictive_table = candidate_table(predictive_candidates).sort_values(
    ["prefix consistent?", "log-likelihood", "accuracy", "number of taps"],
    ascending=[False, False, False, True],
)
save_table(predictive_table, config.metrics_dir / "predictive_summary.csv")
predictive_table.head(30)


,candidate_id,track,method,taps,W,prefix consistent?,log-likelihood,accuracy,number of taps,notes
15,predictive-single_lag_scan-2e0ed784,predictive,single_lag_scan,[63],63,1,-353288.969727,0.500746,1,Simple one-lag agreement baseline.
83,predictive-xgboost-052309e5,predictive,xgboost,"[4, 6, 7, 8, 14, 16, 24, 30, 39, 42, 49, 51, 5...",63,1,-353535.730124,0.500285,14,Tree-based predictive baseline converted into ...
77,predictive-xgboost-b2ef12a7,predictive,xgboost,"[6, 8, 14, 24, 30, 49, 54, 63]",63,1,-353663.269205,0.500047,8,Tree-based predictive baseline converted into ...
81,predictive-xgboost-53092074,predictive,xgboost,"[4, 6, 8, 14, 16, 24, 30, 42, 49, 51, 54, 63]",63,1,-353747.833161,0.499889,12,Tree-based predictive baseline converted into ...
79,predictive-xgboost-16adb438,predictive,xgboost,"[4, 6, 8, 14, 24, 30, 49, 51, 54, 63]",63,1,-353778.331637,0.499832,10,Tree-based predictive baseline converted into ...
76,predictive-xgboost-d3a730d5,predictive,xgboost,"[6, 8, 24, 30, 49, 54, 63]",63,1,-353781.104226,0.499826,7,Tree-based predictive baseline converted into ...
82,predictive-xgboost-be1ded60,predictive,xgboost,"[4, 6, 8, 14, 16, 24, 30, 39, 42, 49, 51, 54, 63]",63,1,-353892.007775,0.499619,13,Tree-based predictive baseline converted into ...
75,predictive-xgboost-018fd5c0,predictive,xgboost,"[6, 24, 30, 49, 54, 63]",63,1,-354154.017409,0.499130,6,Tree-based predictive baseline converted into ...
35,predictive-pair_scan-049380a5,predictive,pair_scan,"[37, 62]",62,1,-354671.005524,0.501580,2,Pairwise XOR interaction screen.
51,predictive-pair_scan-ef4bc68e,predictive,pair_scan,"[2, 62]",62,1,-355048.077590,0.500879,2,Pairwise XOR interaction screen.


This notebook also supports failure analysis. Predictive methods store notes in metadata so the final report can summarize which baselines were weak, which were only useful as priors, and why raw predictive accuracy alone is not enough.